Problem 4: For ./data/ames_prices.csv, define an indicator equal to 1 when Central.Air == 'Y'.
Estimate the proportion of homes with central air and test. Treat a proportion as the
mean of a binary variable and complete the full inference routine.

In [34]:
#importing 
import numpy as np
import pandas as pd
from scipy.stats import norm

#loading data
df = pd.read_csv('./data/ames_prices.csv')
df

,Order,PID,area,price,MS.SubClass,MS.Zoning,Lot.Frontage,Lot.Area,Street,Alley,...,Screen.Porch,Pool.Area,Pool.QC,Fence,Misc.Feature,Misc.Val,Mo.Sold,Yr.Sold,Sale.Type,Sale.Condition
0,1,526301100,1656,215000,20,RL,141.0,31770,Pave,NaN,...,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal
1,2,526350040,896,105000,20,RH,80.0,11622,Pave,NaN,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
2,3,526351010,1329,172000,20,RL,81.0,14267,Pave,NaN,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
3,4,526353030,2110,244000,20,RL,93.0,11160,Pave,NaN,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
4,5,527105010,1629,189900,60,RL,74.0,13830,Pave,NaN,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,1003,142500,80,RL,37.0,7937,Pave,NaN,...,0,0,NaN,GdPrv,NaN,0,3,2006,WD,Normal
2926,2927,923276100,902,131000,20,RL,NaN,8885,Pave,NaN,...,0,0,NaN,MnPrv,NaN,0,6,2006,WD,Normal
2927,2928,923400125,970,132000,85,RL,62.0,10441,Pave,NaN,...,0,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal
2928,2929,924100070,1389,170000,20,RL,77.0,10010,Pave,NaN,...,0,0,NaN,NaN,NaN,0,4,2006,WD,Normal


In [35]:
#central air indicactor
central_air = (df['Central.Air'] == 'Y').astype(int)

central_air

0       1
1       1
2       1
3       1
4       1
       ..
2925    1
2926    1
2927    1
2928    1
2929    1
Name: Central.Air, Length: 2930, dtype: int64

In [3]:
#proportion cal
p_hat = central_air.mean()
print("Proportion with central air:", p_hat)

Proportion with central air: 0.9331058020477816


In [ ]:
#4a:
#2000 bootstrap
rng = np.random.default_rng(100)
R = 2000

results = []

for t in range(R):
    sample = rng.choice(central_air, size=len(central_air), replace=True)
    p_boot = sample.mean()
    results.append(p_boot)

results = np.array(results)

print(results[:5])

[0.93515358 0.93686007 0.93174061 0.93174061 0.93242321]


In [5]:
#4b: 
#bootstrap satndard error
standard_error = np.std(results, ddof=1)

print("Bootstrap Standard Error:", standard_error)

Bootstrap Standard Error: 0.004598110724824475


In [6]:
#4c:
#confidence intervals
ci90 = (p_hat - 1.645 * standard_error,
        p_hat + 1.645 * standard_error)

ci95 = (p_hat - 1.96 * standard_error,
        p_hat + 1.96 * standard_error)

ci99 = (p_hat - 2.576 * standard_error,
        p_hat + 2.576 * standard_error)

print("90% CI:", ci90)
print("95% CI:", ci95)
print("99% CI:", ci99)

90% CI: (np.float64(0.9255419099054454), np.float64(0.9406696941901178))
95% CI: (np.float64(0.9240935050271256), np.float64(0.9421180990684376))
99% CI: (np.float64(0.9212610688206337), np.float64(0.9449505352749294))


In [ ]:
#4d:
#hypothesis testing
p0 = 0.90

z = (p_hat - p0) / standard_error
p_value = 2 * norm.sf(abs(z))

print("Z-statistic:", z)
print("P-value:", p_value)

Z-statistic: 7.1998705618480585
P-value: 6.026974589362344e-13


In [8]:
#4e:
#confidence intervals vs hypothesis test
print("90% CI contains 0.90:", ci90[0] <= p0 <= ci90[1])
print("95% CI contains 0.90:", ci95[0] <= p0 <= ci95[1])
print("99% CI contains 0.90:", ci99[0] <= p0 <= ci99[1])

90% CI contains 0.90: False
95% CI contains 0.90: False
99% CI contains 0.90: False


4f: Interpret the estimate, interval, and test in context. These are observational data: do not turn association into causation.

ANSWER:
- The estimated proportion of homes with central air conditioning is approximately 93.3% 
- The 95% confidence interval is between 0.9241 and 0.9421, this shows us how the population proportion is between these numbers.
- The test demonstrates that the p-value is smaller than 0.10, 0.05, and 0.01, in which the the test we reject the null hypothesis. 

Problem 5: For ./data/ames_prices.csv, estimate the Pearson correlation between arcsinh(area) and
arcsinh(price). Bootstrap paired rows, construct all three CIs, and test . Explain
what would go wrong if area and price were resampled separately.

ANSWER: 
If area and price were resampled seperately that means that the price associated with the original house wouldn't match anymore, since the price and area are sampled togther originally. There would be no correlation for houses with price and area. 

In [9]:
#correlation
area = np.arcsinh(df['area'])
price = np.arcsinh(df['price'])

correlation = area.corr(price)

print("Pearson correlation:", correlation)

Pearson correlation: 0.7231661530091358


In [10]:
#5a: 
#2000 bootstrap
rng = np.random.default_rng(100)
R = 2000

results_corr = []

for t in range(R):
    idx = rng.choice(len(area), size=len(area), replace=True)

    sample_area = area.iloc[idx]
    sample_price = price.iloc[idx]

    corr_boot = sample_area.corr(sample_price)
    results_corr.append(corr_boot)

results_corr = np.array(results_corr)

print(results_corr[:10])

[0.73308653 0.72619263 0.71474197 0.73192828 0.72714561 0.72757466
 0.72096925 0.7070543  0.74262769 0.71840448]


In [11]:
#5b:
#bootstrap standard error
standard_error_corr = np.std(results_corr, ddof=1)

print("Bootstrap Standard Error:", standard_error_corr)

Bootstrap Standard Error: 0.01013295577908988


In [12]:
#5c:
#confidience intervals:
ci90_corr = (correlation - 1.645 * standard_error_corr,
             correlation + 1.645 * standard_error_corr)

ci95_corr = (correlation - 1.96 * standard_error_corr,
             correlation + 1.96 * standard_error_corr)

ci99_corr = (correlation - 2.576 * standard_error_corr,
             correlation + 2.576 * standard_error_corr)

print("90% CI:", ci90_corr)
print("95% CI:", ci95_corr)
print("99% CI:", ci99_corr)

90% CI: (np.float64(0.706497440752533), np.float64(0.7398348652657386))
95% CI: (np.float64(0.7033055596821196), np.float64(0.743026746336152))
99% CI: (np.float64(0.6970636589222002), np.float64(0.7492686470960713))


In [13]:
#5d:
#hypothesis testing:
rho0 = 0

z_corr = (correlation - rho0) / standard_error_corr
p_value_corr = 2 * norm.sf(abs(z_corr))

print("Z-statistic:", z_corr)
print("P-value:", p_value_corr)

Z-statistic: 71.36773995416459
P-value: 0.0


In [14]:
#5e:
#confidence intervals vs hypothesis test
print("90% CI contains 0:", ci90_corr[0] <= rho0 <= ci90_corr[1])
print("95% CI contains 0:", ci95_corr[0] <= rho0 <= ci95_corr[1])
print("99% CI contains 0:", ci99_corr[0] <= rho0 <= ci99_corr[1])

90% CI contains 0: False
95% CI contains 0: False
99% CI contains 0: False


5f: interpreting results 
ANSWER:
- The estimated correlation between transformed house area and house price is approximately 0.7232, showing a positive correlation
- The 95% confidence interval is between 0.7033 and 0.7430, this shows us how the population correlation is between these numbers
- The test demonstrates that the p-value is smaller than 0.10, 0.05, and 0.01, in which the the test we reject the null hypothesis

Problem 6: For ./data/metabric.csv, use Overall Survival (Months) to estimate the mean recorded
time to death or last follow-up, including both living and deceased patients. Let  denote the
population mean of this recorded-time variable, and test  months. Complete the full
inference routine and interpret the CIs in both months and years. Living patients' eventual death
times are right-censored, so this exercise does not estimate unrestricted mean lifetime. Explain
that distinction in your interpretation.

In [15]:
#loading data
metabric = pd.read_csv("./data/metabric.csv")

metabric.head()

,Age at Diagnosis,Type of Breast Surgery,Cancer Type,Chemotherapy,Hormone Therapy,Lymph nodes examined positive,Mutation Count,Nottingham prognostic index,Overall Survival (Months),Overall Survival Status,Radio Therapy,TMB (nonsynonymous),Tumor Size,Tumor Stage
0,43.19,BREAST CONSERVING,Breast Cancer,NO,YES,0.0,2.0,4.020,84.633333,0:LIVING,YES,2.615035,10.0,1.0
1,48.87,MASTECTOMY,Breast Cancer,YES,YES,1.0,2.0,4.030,163.700000,1:DECEASED,NO,2.615035,15.0,2.0
2,47.68,MASTECTOMY,Breast Cancer,YES,YES,3.0,1.0,4.050,164.933333,0:LIVING,YES,1.307518,25.0,2.0
3,76.97,MASTECTOMY,Breast Cancer,YES,YES,8.0,2.0,6.080,41.366667,1:DECEASED,YES,2.615035,40.0,2.0
4,78.77,MASTECTOMY,Breast Cancer,NO,YES,0.0,4.0,4.062,7.800000,1:DECEASED,YES,5.230071,31.0,4.0


In [16]:
#6a: 
#mean:
survival = metabric['Overall Survival (Months)'].dropna()

mean_survival = survival.mean()

print("Mean survival time:", mean_survival)

Mean survival time: 129.09429138744105


In [17]:
#2000 bootstraps
rng = np.random.default_rng(100)
R = 2000

results_survival = []

for t in range(R):
    sample = rng.choice(survival, size=len(survival), replace=True)
    mean_boot = sample.mean()
    results_survival.append(mean_boot)

results_survival = np.array(results_survival)

print(results_survival[:10])

[130.54211963 132.12742616 126.81404815 124.11176471 129.26942169
 130.38423927 129.79128816 130.62345495 127.61514023 125.53248945]


In [18]:
#6b:
#bootstrap SE:
standard_error_survival = np.std(results_survival, ddof=1)

print("Bootstrap Standard Error:", standard_error_survival)

Bootstrap Standard Error: 2.089004448845054


In [19]:
#6c: 
#confidence intervals:
ci90_survival = (mean_survival - 1.645 * standard_error_survival,
                 mean_survival + 1.645 * standard_error_survival)

ci95_survival = (mean_survival - 1.96 * standard_error_survival,
                 mean_survival + 1.96 * standard_error_survival)

ci99_survival = (mean_survival - 2.576 * standard_error_survival,
                 mean_survival + 2.576 * standard_error_survival)

print("90% CI (months):", ci90_survival)
print("95% CI (months):", ci95_survival)
print("99% CI (months):", ci99_survival)

90% CI (months): (np.float64(125.65787906909094), np.float64(132.53070370579115))
95% CI (months): (np.float64(124.99984266770474), np.float64(133.18874010717735))
99% CI (months): (np.float64(123.71301592721619), np.float64(134.4755668476659))


In [20]:
#month to years
print("90% CI (years):", np.array(ci90_survival) / 12)
print("95% CI (years):", np.array(ci95_survival) / 12)
print("99% CI (years):", np.array(ci99_survival) / 12)

90% CI (years): [10.47148992 11.04422531]
95% CI (years): [10.41665356 11.09906168]
99% CI (years): [10.30941799 11.20629724]


In [21]:
#6d: 
#hypothesis testing:
mu0 = 120

z_survival = (mean_survival - mu0) / standard_error_survival
p_value_survival = 2 * norm.sf(abs(z_survival))

print("Z-statistic:", z_survival)
print("P-value:", p_value_survival)

Z-statistic: 4.353409296217152
P-value: 1.3403656666455162e-05


In [22]:
#6e:
#confidence intervals vs hypothese tests
print("90% CI contains 120:", ci90_survival[0] <= mu0 <= ci90_survival[1])
print("95% CI contains 120:", ci95_survival[0] <= mu0 <= ci95_survival[1])
print("99% CI contains 120:", ci99_survival[0] <= mu0 <= ci99_survival[1])

90% CI contains 120: False
95% CI contains 120: False
99% CI contains 120: False


6f: interpret intervals
ANSWER:
- The estimated mean is 129.09 months
- The 95% confidence interval is between 125.00  and 133.19 for months this shows us how the population mean is between these numbers
- The test demonstrates that the p-value is smaller than 0.10, 0.05, and 0.01, in which the the test we reject the null hypothesis, thid differs from the recorded tiem for 120 months
- Patients are not defined by this average survival time because there are patients that are still alive

Problem 7: For ./data/nhanes_data_17_18.csv, use AgeInYearsAtScreening to select respondents
aged 18 or older with observed SleepHoursWeekdaysOrWorkdays. Estimate mean weekday
sleep and test  hours. Comment on whether statistical and practical significance tell the
same story. For this exercise, treat observations as independent and unweighted, ignoring the
complex survey design. Do not interpret these results as nationally representative population
estimates or survey-adjusted confidence intervals.

In [24]:
#loading data
nhanes = pd.read_csv("./data/nhanes_data_17_18.csv")

nhanes.head()

/var/folders/90/1v01rlw511b4d8c3kfjvlv2r0000gn/T/ipykernel_1326/961050364.py:2: DtypeWarning: Columns (142) have mixed types. Specify dtype option on import or set low_memory=False.
  nhanes = pd.read_csv("./data/nhanes_data_17_18.csv")


,SEQN,GeneralHealthCondition,EverBreastfedOrFedBreastmilk,AgeStoppedBreastfeedingdays,AgeFirstFedFormuladays,AgeStoppedReceivingFormuladays,AgeStartedOtherFoodbeverage,AgeFirstFedMilkdays,TypeOfMilkFirstFedWholeMilk,TypeOfMilkFirstFed2Milk,...,DaysSmokedCigsDuringPast30Days,AvgCigarettesdayDuringPast30Days,TriedToQuitSmoking,TimesStoppedSmokingCigarettes,HowLongWereYouAbleToStopSmoking,UnitOfMeasureDayweekmonthyear_2_SMQ,CurrentSelfreportedHeightInches,CurrentSelfreportedWeightPounds,TriedToLoseWeightInPastYear,TimesLost10LbsOrMoreToLoseWeight
0,93703.0,NaN,1.0,273.0,1.0,365.0,152.0,365.0,Whole or regular milk,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704.0,NaN,1.0,60.0,3.0,365.0,126.0,365.0,Whole or regular milk,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,93705.0,Good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,63.0,165.0,0.0,11 times or more
3,93706.0,Very good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,68.0,145.0,0.0,Never
4,93707.0,Good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
#filtering
adults = nhanes[nhanes['AgeInYearsAtScreening'] >= 18]

sleep = adults['SleepHoursWeekdaysOrWorkdays'].dropna()

print("Number of adults:", len(sleep))
sleep.head()

Number of adults: 5489


2     8.0
3    10.5
5     8.0
6     7.0
7     7.0
Name: SleepHoursWeekdaysOrWorkdays, dtype: float64

In [26]:
#7a:
#mean:
mean_sleep = sleep.mean()

print("Mean weekday sleep hours:", mean_sleep)

Mean weekday sleep hours: 7.62124248496994


In [27]:
#2000 bootstrap:
rng = np.random.default_rng(100)
R = 2000

results_sleep = []

for t in range(R):
    sample = rng.choice(sleep, size=len(sleep), replace=True)
    mean_boot = sample.mean()
    results_sleep.append(mean_boot)

results_sleep = np.array(results_sleep)

print(results_sleep[:10])

[7.60284205 7.62188012 7.67334669 7.61932957 7.67097832 7.65695026
 7.63590818 7.59118236 7.62260885 7.62515941]


In [28]:
#7b:
#bootstrap SE:
standard_error_sleep = np.std(results_sleep, ddof=1)

print("Bootstrap Standard Error:", standard_error_sleep)

Bootstrap Standard Error: 0.022184158284392137


In [29]:
#7c:
#confidence intervals
ci_90_sleep = (
    mean_sleep - 1.645 * standard_error_sleep,
    mean_sleep + 1.645 * standard_error_sleep
)

ci_95_sleep = (
    mean_sleep - 1.96 * standard_error_sleep,
    mean_sleep + 1.96 * standard_error_sleep
)

ci_99_sleep = (
    mean_sleep - 2.576 * standard_error_sleep,
    mean_sleep + 2.576 * standard_error_sleep
)

print("90% CI:", ci_90_sleep)
print("95% CI:", ci_95_sleep)
print("99% CI:", ci_99_sleep)

90% CI: (np.float64(7.584749544592115), np.float64(7.657735425347766))
95% CI: (np.float64(7.577761534732532), np.float64(7.664723435207349))
99% CI: (np.float64(7.564096093229346), np.float64(7.678388876710534))


In [ ]:
#7d
#hypothesis testing

null_mean_sleep = 8

z_sleep = (mean_sleep - null_mean_sleep) / standard_error_sleep
p_value_sleep = 2 * norm.sf(abs(z_sleep))

print("Z-score:", z_sleep)
print("P-value:", p_value_sleep)

Z-score: -17.073332698700494
P-value: 2.344273639198169e-65


In [31]:
print("At α = 0.10:", p_value_sleep < 0.10)
print("At α = 0.05:", p_value_sleep < 0.05)
print("At α = 0.01:", p_value_sleep < 0.01)

At α = 0.10: True
At α = 0.05: True
At α = 0.01: True


In [32]:
#7e:
#confidence intervals:
print("90% CI contains 8:", ci_90_sleep[0] <= 8 <= ci_90_sleep[1])
print("95% CI contains 8:", ci_95_sleep[0] <= 8 <= ci_95_sleep[1])
print("99% CI contains 8:", ci_99_sleep[0] <= 8 <= ci_99_sleep[1])

90% CI contains 8: False
95% CI contains 8: False
99% CI contains 8: False


7f: interpreting results
ANSWER: 
- The estimated mean is 7.62 hours
- The 95% confidence interval is between 7.58 and 7.66 hours this shows us how the population mean is between these numbers
- The test demonstrates that the p-value is smaller than 0.10, 0.05, and 0.01, in which the the test we reject the null hypothesis
- Comment on whether statistical and practical significance tell the same story: There is statistically a significance between the tested mean and original mean, it does not mean that is it also practically signficant. 